# W08A Colab A · 평가지표·MF·SGD

딥러닝응용I(추천시스템) · 동덕여자대학교 · 유원상 교수 · 2026년 10월 20일

CPU에서 위부터 실행합니다. A/B는 각각 독립 실행됩니다. GPU·유료 API·필수 파일 업로드가 필요하지 않습니다. **파일 → Drive에 사본 저장**으로 개인 실습본을 만드세요. 설치가 런타임 재시작을 요구하면 재시작 후 첫 셀부터 실행합니다.

실제 5~6주차를 복습합니다. 설명용 수치는 강의노트의 worked example과 같으며 60개 연습문제의 입력·정답과 구별합니다. Python 문제는 **A에 12개, B에 9개**로 나누고 QID를 유지합니다. 전체 실행 성공과 자신의 문제 풀이 완료는 다릅니다.

학습 순서는 관측과 분모 확인 → 예측과 손실 연결 → old 상태의 갱신 → 검증 선택 → 재구성의 의미입니다. 필요한 중간 활동과 앱 기록은 Q06/Q10/Q18/Q26/Q31/Q39/Q48/Q51/Q58에 연결합니다.


## 코드의 입력·상태·출력을 읽는 방법

표의 함수 이름을 외우는 것이 목표가 아닙니다. 아래 정의 링크에서 **입력 조건 → 계산 → 반환값**을 확인합니다. 새 W08A 함수는 기존 함수를 연결하며 학습 알고리즘을 다시 구현하지 않습니다.

| API | 입력 → 출력과 역할 |
|---|---|
| `rating_metrics` | 같은 길이의 실제·예측 1D 배열 → `n`, `mae`, `rmse`; clipping 없음 |
| `sgd_step` | 같은 길이의 old p/q, 한 평점, 평균·편향·α·β → old 예측/잔차/손실과 새 모수; 입력 불변 |
| `MFSGD.fit` | 관측된 user_id/movie_id/rating 표 → 학습된 모델; 미관측은 행에서 제외 |
| `fit_validated` | 겹치지 않는 train/validation 표와 설정 → model/history/best_epoch; 상한까지 학습 후 모든 최적 모수 복원 |
| `predict_rows` | 학습된 모델과 관측 ID 표 → 입력 행 순서의 예측 배열; 모델 변경 없음 |
| `svd_steps` | NaN이 없는 완전한 실수 행렬과 rank → U/s/Vt/Gram/근사/오차; rank 0 허용 |
| `metric_review_demo` | 선택 예측 4개 → 설명 예제의 관측별 표와 지표 |
| `sgd_review_demo` | α·β → 동일 old 상태에서 한 번의 갱신과 비교표 |
| `mf_review_data` | 인자 없음 → 새 합성 train15/validation3 표와 별도 2×2 완전 행렬 |
| `mf_review_experiment` | K·β·epoch 상한·rank → MF 검증·복원 상태와 별도 SVD 결과 dict |
| `mf_review_view` | 설정과 선택 report_fn → 설명, history, 검증예측표, 재구성표, Figure |
| `build_mf_review_app` | 선택 report_fn → 새 Gradio Blocks; 이 단계에서는 서버 시작 없음 |
| `build_svd_app` | 선택 해석 함수 → 이전 W06B의 SVD·MF 누적 앱 |
| `definition_link` | 실제 함수·클래스·메소드와 고정 ref → 그 버전의 GitHub 정의 시작 행 |

`fit`은 상태를 만드는 단계이고 예측은 그 상태를 읽는 단계입니다. validation은 선택에 쓰지만 gradient에는 넣지 않습니다. `fit_validated`는 실제 조기 종료를 하지 않습니다. 사용자 수 M·아이템 수 N·잠재요인 수 K·추천 길이 N_rec·SVD rank r를 구별합니다.


### 실행 환경 준비

고정 버전의 공통 패키지를 가져옵니다. 설치 ref와 아래 정의 링크의 ref는 동일합니다. 로컬 검증에서는 준비된 프로젝트 환경을 사용합니다.


In [ ]:
import os, sys, subprocess, math, base64
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_VERSION = '2026-fall-w08a'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'], check=True)
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from luna_recsys.mf_sgd import MFSGD, sgd_step
from luna_recsys.mf_evaluation import rating_metrics, fit_validated, predict_rows
from luna_recsys.svd_foundations import svd_steps
from luna_recsys.svd_lab import build_svd_app
from luna_recsys.mf_review import (metric_review_demo, sgd_review_demo, mf_review_data,
    mf_review_experiment, mf_review_view, build_mf_review_app)
from luna_recsys.model_comparison import definition_link
print('Python:', sys.version.split()[0], '| 수업 설치 버전:', COURSE_VERSION)


### 정의 위치 확인

각 링크는 실제 import한 함수·클래스·메소드의 선언 시작 행을 가리킵니다. 입력 규약은 `help(함수)`로도 확인할 수 있습니다.


In [ ]:
apis = [rating_metrics, sgd_step, MFSGD, MFSGD.fit, fit_validated,
        predict_rows, svd_steps, metric_review_demo, sgd_review_demo, mf_review_data,
        mf_review_experiment, mf_review_view, build_mf_review_app, build_svd_app,
        definition_link]
display(Markdown('\n'.join(
    f'- [{fn.__qualname__}]({definition_link(fn, COURSE_VERSION)})' for fn in apis)))


## 계산 전에 보는 관계도

아래 그림은 이 수업을 위해 만든 설명용 도식입니다. QID 연결에 따라 기호·차원·분모를 읽습니다. 그림 바이트를 notebook에 포함하여 Colab의 상대경로에 의존하지 않습니다.


**설명 그림: 01-metrics-denominators** · 강의노트의 같은 그림과 대조합니다.


In [ ]:
display(Image(data=base64.b64decode('...'), width=900))


**설명 그림: 03-factor-shapes** · 강의노트의 같은 그림과 대조합니다.


In [ ]:
display(Image(data=base64.b64decode('...'), width=900))


**설명 그림: 05-sgd-old-state** · 강의노트의 같은 그림과 대조합니다.


In [ ]:
display(Image(data=base64.b64decode('...'), width=900))


## 1. 같은 관측, 다른 오차 요약 · Q06/Q10 연결

설명 예제의 실제값은 (5,3,2,4), 예측은 (4,3,3,3)입니다. 잔차는 실제−예측으로 정의합니다. 관측별 절댓값과 제곱을 만든 다음 **평가 관측 수 n=4**로 나눕니다. RMSE에는 마지막 제곱근이 필요합니다.

$$
\mathrm{MAE}=\frac{1}{n}\sum_{j=1}^n|y_j-\hat y_j|,\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{j=1}^n(y_j-\hat y_j)^2}.
$$

아래 표는 중간 계산을 생략하지 않습니다. 이 예제의 MAE는 0.75, RMSE는 약 0.866025입니다. 평점 오차와 추천 목록의 Precision/Recall은 서로 다른 질문에 답합니다. 순위 지표 문제의 후보·관련집합·목록 길이 N_rec 조건을 다시 읽습니다.


In [ ]:
metric_demo = metric_review_demo()
display(metric_demo['table'])
print(metric_demo['metrics'])


## 2. MF 행렬과 관측 표 · Q18 연결

한 사용자와 한 영화의 예측은 평균·두 편향·두 잠재벡터의 내적을 더합니다. M은 사용자 수, N은 영화 수이고 P의 크기는 M×K, Q의 크기는 N×K입니다. 아래 행렬식의 B와 C는 각각 사용자 편향을 열 방향으로, 영화 편향을 행 방향으로 반복한 M×N 행렬입니다. 즉 B의 (u,i) 원소는 b_u이고 C의 (u,i) 원소는 c_i이며 평균 μ도 모든 칸에 더합니다.

$$
\hat r_{ui}=\mu+b_u+c_i+p_u^\top q_i,\qquad \hat R=\mu+B+C+PQ^\top.
$$

`mf_review_data()`는 18개 합성 관측 중 15개를 train, 3개를 validation으로 나눈 사본을 반환합니다. 보류 관측의 정답은 데이터 생성에서 이미 알려져 있으나 학습 함수에는 train만 전달합니다. 원래 미관측인 칸에 정답 0을 만들어 넣지 않습니다. 다음 셀은 관측 행을 원표와 마스크로 바꾸어 봅니다.


In [ ]:
demo_data = mf_review_data()
display(demo_data['train'])
display(demo_data['validation'])
train_matrix = demo_data['train'].pivot(index='user_id', columns='movie_id', values='rating')
display(train_matrix)
display(train_matrix.notna().astype(int))
print('train / validation:', len(demo_data['train']), len(demo_data['validation']))


## 3. old 상태에서 한 번 갱신 · Q26/Q31 연결

설명 조건은 K=1, μ=3, b=0.1, c=−0.1, p=0.5, q=0.4, 실제 평점 r=4, α=0.1, β=0.1입니다. 갱신 전 예측은 3.2이고 잔차는 0.8입니다. 이번 수업은 **한 관측마다** 다음 손실과 정규화를 적용합니다.

$$
L_{ui}=\frac12 e_{ui}^2+\frac{\beta}{2}(\|p_u\|^2+\|q_i\|^2+b_u^2+c_i^2).
$$

$$
p^{new}=p^{old}+\alpha(e q^{old}-\beta p^{old}),\qquad
q^{new}=q^{old}+\alpha(e p^{old}-\beta q^{old}).
$$

`sgd_review_demo`는 기존 `sgd_step`을 호출하여 p/q/b/c의 old와 new를 나란히 보여 줍니다. 두 요인의 새 값은 0.527과 0.436입니다. q 계산에 0.527을 다시 사용하면 이번 규약의 동시 갱신과 달라집니다. 학습률이 크면 한 단계 손실이 감소하지 않을 수도 있습니다.


In [ ]:
step_demo = sgd_review_demo(alpha=.1, beta=.1)
display(step_demo['table'])
print({key: step_demo['step'][key] for key in ['prediction', 'error', 'loss', 'prediction_new', 'loss_new']})


## 강의자료와 같은 번호의 Python 연습

문제 코드는 Markdown 안에 두어 의도한 오류나 빈칸이 전체 실행을 중단시키지 않게 했습니다. 문제 바로 아래 실행 셀에 자신의 코드를 씁니다. 안전한 starter의 실행 성공은 문제를 풀었다는 뜻이 아닙니다. 문제 원문·수치·소질문은 강의자료와 동일합니다.


### Q09 · 빈칸: 오차의 계산 순서

다음 빈칸 두 개를 채우고 출력될 MAE와 RMSE를 소수 여섯째 자리까지 예측하라. `sum`은 합, `len`은 개수, `x ** 0.5`는 제곱근이다.

```python
actual = [4, 1, 5]
predicted = [3, 2, 3]
errors = [a - p for a, p in zip(actual, predicted)]
mae = sum(abs(e) for e in errors) / ____
rmse = (sum(e * e for e in errors) / ____) ** 0.5
print(round(mae, 6), round(rmse, 6))
```


In [ ]:
# Q09 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q09_my_answer = None


### Q10 · 오류 찾기: 뒤바뀐 순위 지표 분모

다음 코드는 요청한 Top-6에서 후보 부족으로 네 편만 반환한 상황이다. 관련성은 보류 자료에서 확인된 영화만 사용한다. (가) 두 지표의 분모 오류를 고쳐라. (나) 수정 전후 출력을 쓰고, Precision 분모를 `len(recommended)`로 고치는 것도 이 수업 규약과 다른 이유를 설명하라.

```python
recommended = ['A', 'B', 'C', 'D']
relevant = {'A', 'C', 'E', 'F', 'G'}
N_rec = 6
hits = sum(item in relevant for item in recommended)
precision = hits / len(relevant)
recall = hits / N_rec
print(round(precision, 6), round(recall, 6))
```


In [ ]:
# Q10 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q10_my_answer = None


### Q11 · 빈칸: 실제 순위와 이상 순위

Top-3의 적중 여부가 `hits=[0,1,1]`이고 관련 영화는 총 두 편이다. `math.log2(x)`는 밑이2인 로그이며 위치 번호는1부터 센다. 두 빈칸을 채우고 NDCG를 구하라.

```python
import math
hits = [0, 1, 1]
weights = [1 / math.log2(j + 1) for j in range(1, 4)]
dcg = sum(h * w for h, w in zip(hits, weights))
ideal_hits = [____, ____, 0]
idcg = sum(h * w for h, w in zip(ideal_hits, weights))
print(round(dcg / idcg, 6))
```


In [ ]:
# Q11 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q11_my_answer = None


### Q12 · 실행 추적: 제외와 macro 평균

각 튜플은 `(적중 수, 관련 영화 수)`이다. 다음 코드의 출력 네 값을 순서대로 쓰고 마지막 두 값이 다른 이유를 설명하라. `sum`은 합, `len`은 개수다.

```python
rows = [(1, 1), (1, 3), (0, 0)]
valid = [(h, g) for h, g in rows if g > 0]
macro = sum(h / g for h, g in valid) / len(valid)
pooled = sum(h for h, g in valid) / sum(g for h, g in valid)
print(len(valid), len(rows) - len(valid), round(macro, 6), round(pooled, 6))
```


In [ ]:
# Q12 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q12_my_answer = None


### Q20 · 빈칸: 예측·잔차·목적함수

두 빈칸에 각각 내적 표현과 잔차 표현을 채우고 예측과 전체 손실을 구하라. `zip`은 대응 원소를 묶고 `sum`은 더한다.

```python
p = [0.2, 0.5]
q = [0.5, 0.2]
mu, b, c, rating, beta = 3.0, 0.1, -0.2, 4.0, 0.2
interaction = ____
prediction = mu + b + c + interaction
error = ____
penalty = beta / 2 * (sum(x*x for x in p + q) + b*b + c*c)
loss = 0.5 * error**2 + penalty
print(round(prediction, 6), round(loss, 6))
```


In [ ]:
# Q20 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q20_my_answer = None


### Q21 · 오류 찾기: 미관측을 학습 정답으로 넣기

`None`은 평점이 없다는 뜻이다. 다음 코드에서 관측된 평점만을 평가하는 MSE로 고쳐라. (가) 잘못 추가된 정답과 분모 오류를 설명하라. (나) 수정 전후 출력과 평가 관측 수를 쓰라. 여기서는 1/2을 곱한 학습 손실이 아니라 평균제곱오차(MSE)를 구한다.

```python
ratings = [4, None, 2]
predicted = [3, 2, 3]
filled = [0 if r is None else r for r in ratings]
mse = sum((r-p)**2 for r,p in zip(filled,predicted)) / len(filled)
print(round(mse, 6))
```


In [ ]:
# Q21 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q21_my_answer = None


### Q22 · 실행 추적: 한 사용자와 한 영화 선택

P의 행은 사용자, Q의 행은 영화이며 Python 인덱스는0부터 시작한다. (가) 코드의 두 출력 줄을 쓰라. (나) 계산한 대상은 몇 번째 사용자·몇 번째 영화이며 전체 예측 행렬은 어떤 크기인가?

```python
P = [[0.2, 0.4], [0.3, -0.1]]
Q = [[0.1, 0.2], [0.5, 0.5], [-0.2, 0.4]]
user_bias = [0.1, -0.1]
item_bias = [0.0, 0.2, -0.2]
mu = 3.0
u, i = 0, 1
parts = [a*b for a,b in zip(P[u],Q[i])]
prediction = mu + user_bias[u] + item_bias[i] + sum(parts)
print(len(P), len(Q), len(P[0]))
print([round(x, 6) for x in parts], round(prediction, 6))
```


In [ ]:
# Q22 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q22_my_answer = None


### Q30 · 빈칸: old 상태로 두 벡터 갱신

갱신 전 잔차는 이미 `error=0.8`로 계산되었다. 다음 코드의 빈칸 두 개에 표현식을 넣고 두 출력 리스트를 쓰라. `p_old`와 `q_old`는 갱신 전 복사본이며 `zip`은 같은 위치를 묶는다.

```python
p_old = [0.3, -0.1]
q_old = [0.2, 0.4]
error, alpha, beta = 0.8, 0.1, 0.2
p_new = [p + alpha * (____) for p,q in zip(p_old,q_old)]
q_new = [q + alpha * (____) for p,q in zip(p_old,q_old)]
print([round(x, 6) for x in p_new])
print([round(x, 6) for x in q_new])
```


In [ ]:
# Q30 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q30_my_answer = None


### Q31 · 오류 찾기: 새 p를 즉시 재사용

수업의 기본 규약은 갱신 전 p·q와 같은 잔차로 두 모수를 갱신하는 것이다. 아래 잔차1.0은 갱신 전에 계산된 값이며 β=0이다. (가) 이 규약을 어긴 줄과 이유를 찾고 코드를 고쳐라. (나) 원래 출력과 수정 출력을 쓰라. 잔차는 다시 계산하지 않는다.

```python
p, q = 0.4, 0.5
error, alpha = 1.0, 0.1
p = p + alpha * error * q
q = q + alpha * error * p
print(round(p, 6), round(q, 6))
```


In [ ]:
# Q31 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q31_my_answer = None


### Q32 · 빈칸: 평균과 두 편향의 역할

갱신 전 내적은0.1이며 μ는 학습 자료에서 정한 고정 평균이다. 다음 두 빈칸을 채워 두 편향을 같은 잔차로 갱신하고 출력을 쓰라. 벡터 갱신은 이번 코드에서 제외한다.

```python
mu, b_old, c_old = 3.0, 0.2, -0.1
interaction, rating = 0.1, 4.0
alpha, beta = 0.05, 0.2
error = rating - (mu + b_old + c_old + interaction)
b_new = b_old + alpha * (____)
c_new = c_old + alpha * (____)
print(round(mu, 6), round(b_new, 6), round(c_new, 6))
```


In [ ]:
# Q32 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q32_my_answer = None


### Q33 · 오류 찾기: 기울기를 더한 코드

이 문항은 q와μ를 고정하고 p 한 좌표만 갱신하는 실험이다. β=0이고 손실은 `0.5 * error**2`다. 다음 코드의 오류를 고쳐라. (가) 기울기와 갱신 방향을 설명하라. (나) 수정 전후의 p와 새 손실을 쓰고 원래 손실0.5와 비교하라.

```python
p, q, mu, rating = 0.4, -0.5, 3.2, 4.0
alpha = 0.1
error = rating - (mu + p*q)
gradient = -error*q
p_new = p + alpha*gradient
loss_new = 0.5*(rating-(mu+p_new*q))**2
print(round(p_new, 6), round(loss_new, 6))
```


In [ ]:
# Q33 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q33_my_answer = None


### Q34 · 실행 추적: 영벡터와 편향

다음은 K=1, 두 잠재벡터와 두 편향을0으로 시작한 한 관측의 동시 갱신이다. (가) 출력 다섯 값을 구하라. (나) ‘영초기화면 모든 모수가 전혀 학습되지 않는다’는 문장을 고쳐라. 모든 우변은 old값이다.

```python
p_old = q_old = b_old = c_old = 0.0
mu, rating, alpha, beta = 3.0, 4.0, 0.1, 0.2
error = rating - (mu+b_old+c_old+p_old*q_old)
p_new = p_old + alpha*(error*q_old-beta*p_old)
q_new = q_old + alpha*(error*p_old-beta*q_old)
b_new = b_old + alpha*(error-beta*b_old)
c_new = c_old + alpha*(error-beta*c_old)
pred_new = mu+b_new+c_new+p_new*q_new
print(*(round(x, 6) for x in [p_new,q_new,b_new,c_new,pred_new]))
```


In [ ]:
# Q34 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q34_my_answer = None


## 점검 퀴즈

강의노트에 있는 동일한 다섯 확인 항목입니다. 새로운 독립 문항을 추가하지 않습니다.

1. 설명 예제의 MAE와 RMSE가 원래 평점 단위로 해석되는 이유는 무엇인가요? (연결 Q06)
2. 설명 예제의 MF 예측에서 사용자·아이템 편향은 각각 어떤 기준 차이를 나타내나요? (연결 Q18)
3. 설명 예제에서 p를 먼저 갱신한 뒤 새 p로 q를 계산하면 명시된 동시 갱신과 같을까요? (연결 Q31)
4. 설명용 검증 곡선에서 선택 epoch의 번호만 저장하면 충분한가요? (연결 Q39)
5. 설명 예제의 SVD 에너지 보존율 90%를 추천 정확도라고 부를 수 있나요? (연결 Q58)


## 실행 검증과 정리

아래 검사는 설명 예제의 분모·old 갱신·데이터 경계를 확인합니다. 자신의 연습문제 답을 검사하거나 공개 정답을 제공하는 셀이 아닙니다. B에서는 같은 원리를 검증·SVD·앱으로 연결합니다.


In [ ]:
assert metric_demo['metrics']['n'] == 4
assert np.isclose(metric_demo['metrics']['mae'], .75)
assert np.isclose(metric_demo['metrics']['rmse'], np.sqrt(.75))
np.testing.assert_allclose(step_demo['table']['new'], [.527, .436, .179, -.019])
train_pairs = set(demo_data['train'][['user_id','movie_id']].itertuples(index=False, name=None))
val_pairs = set(demo_data['validation'][['user_id','movie_id']].itertuples(index=False, name=None))
assert not train_pairs & val_pairs
print('W08A_A_RUNTIME_CHECK_PASS: 설명 예제와 관측 경계 확인')


## 참고자료와 다음 복습

- [이번 강의노트](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/sessions/w08a-mf-svd-review.md)
- [W05A 지표 보강 S1–S5](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/supplements/w05a-metrics-remediation.md)
- [W05B SGD 워크북 S1–S3](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/supplements/w05b-sgd-workbook.md)
- [W06A 검증 워크북 W1–W3](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/supplements/w06a-evaluation-workbook.md)
- [W06B SVD 계산 복습](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/supplements/w06b-calculation-practice.md)
- [NumPy SVD 공식 문서](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)
- [scikit-learn 데이터 누수 설명](https://scikit-learn.org/stable/common_pitfalls.html)

필수 실습은 작은 합성 자료만 사용합니다. 실제 MovieLens 자동 준비·캐시 실험은 기존 W05A/W06A의 선택 확장 경로를 이용합니다. 서로 다른 표본의 점수를 같은 통제 실험처럼 비교하지 않습니다.
